# DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

As of this whole code is commented out please login and get a OPEN AI API key and uncomment the lines to run the code.

In [1]:
# # importing Libraries

# import os
# import re
# import json
# from dotenv import load_dotenv
# from huggingface_hub import login
# from openai import OpenAI
# from pricer.items  import Item
# from pricer.evaluator import evaluate

In [2]:
# # environment

# LITE_MODE = False
# # PLease go with smaller dataset because the dataset size will cost you more

# load_dotenv(override=True)
# hf_token = os.environ[<your_HF_token>]
# login(hf_token, add_to_git_credential=True)

In [3]:
# username = "Arivukkarasu"
# dataset = f"{username}/Amazon_items_lite" if LITE_MODE else f"{username}/Amazon_items_full"

# train, val, test = Item.from_hub(dataset)

# print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [5]:
# openai = OpenAI(api_key="your_api_key_here")

### Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

For 20,000 points cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [6]:
# # OpenAI recommends fine-tuning with populations of 50-100 examples
# # But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


# fine_tune_train = train[:100]
# fine_tune_validation = val[:50]

In [7]:
# len(fine_tune_train)

### Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [8]:
# def messages_for(item):
#     message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
#     return [
#         {"role": "user", "content": message},
#         {"role": "assistant", "content": f"${item.price:.2f}"}
#     ]

In [9]:
# messages_for(fine_tune_train[0])

In [10]:
# # Convert the items into a list of json objects - a "jsonl" string
# # Each row represents a message in the form:
# # {"messages" : [{"role": "system", "content": "You estimate prices...


# def make_jsonl(items):
#     result = ""
#     for item in items:
#         messages = messages_for(item)
#         messages_str = json.dumps(messages)
#         result += '{"messages": ' + messages_str +'}\n'
#     return result.strip()

In [11]:
# print(make_jsonl(train[:3]))

In [12]:
# # Convert the items into jsonl and write them to a file

# def write_jsonl(items, filename):
#     with open(filename, "w") as f:
#         jsonl = make_jsonl(items)
#         f.write(jsonl)

In [13]:
# write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
# write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [14]:
# with open("jsonl/fine_tune_train.jsonl", "rb") as f:
#     train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
# train_file

In [16]:
# with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
#     validation_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
# validation_file

https://platform.openai.com/storage/files/

This will show the status of the files and fine-tuning in Open AI Dashboard.

### Step 2

#### And now time to Fine-tune!

In [18]:
# openai.fine_tuning.jobs.create(
#     training_file=train_file.id,
#     validation_file=validation_file.id,
#     model="gpt-4.1-nano-2025-04-14",
#     seed=42,
#     hyperparameters={"n_epochs": 1, "batch_size": 1},
#     suffix="pricer"
# )

In [19]:
# openai.fine_tuning.jobs.list(limit=1)

In [20]:
# job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [21]:
# job_id

In [22]:
# openai.fine_tuning.jobs.retrieve(job_id)

In [23]:
# openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

### Step 3

Test our fine tuned model

In [24]:
# fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [25]:
# fine_tuned_model_name

In [26]:
# # The prompt

# def test_messages_for(item):
#     message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
#     return [
#         {"role": "user", "content": message},
#     ]

In [27]:
# # Try this out

# test_messages_for(test[0])

In [28]:
# # The inference function


# def gpt_4__1_nano_fine_tuned(item):
#     response = openai.chat.completions.create(
#         model=fine_tuned_model_name,
#         messages=test_messages_for(item),
#         max_tokens=7
#     )
#     return response.choices[0].message.content

In [29]:
# print(test[0].price)
# print(gpt_4__1_nano_fine_tuned(test[0]))

In [30]:
# evaluate(gpt_4__1_nano_fine_tuned, test)

In [31]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000